Goal: Building CLI chatbot that remembers the conversation

In [4]:
# pip show langgraph

1. import libraries

In [75]:
# from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
# from langchain_core.runnables import RunnablePassthrough
# from langchain_core.runnables import RunnableMap


2. build the model to use upstream

In [76]:
# openai key def
from langchain_openai import AzureChatOpenAI
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
az_aoai_key = os.getenv("az_aoai_key")
az_model = os.getenv("az_model")
az_endpoint = os.getenv("az_endpoint")
api_ver = os.getenv("api_ver")

# print(az_aoai_key)

chat_model = AzureChatOpenAI(
    api_key=az_aoai_key,  # type: ignore
    api_version=api_ver,
    model=az_model,
    azure_endpoint=az_endpoint,
    temperature=0.01
)

3. craft the propmt to use

In [77]:
sys_prompt = ChatPromptTemplate.from_messages([  # type: ignore
    ("system", "You are a helpful support assistant for answering user questions."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

4. create chain using LCEL

In [78]:
# udf_chain = (
#     RunnableMap({
#         "input": lambda x: x["input"],
#         "history": lambda x: x.get("history", [])
#     })
# | sys_prompt | chat_model
# )


udf_chain = sys_prompt | chat_model  # type: ignore


5. storing different session of user interaction

In [66]:
from typing import Dict
store_session_history = {}

def get_session_history(session_id: str) -> Dict[str, str]:
    if session_id not in store_session_history:
        store_session_history[session_id] = ChatMessageHistory()
    return store_session_history[session_id]  # type: ignore

5B. let store the user interaction in json based file instead of session

In [79]:
import json
import os
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import BaseMessage, message_to_dict, messages_from_dict    # type: ignore

SAVE_FILENAME = "chat_session.json"

def load_store():   # type: ignore
    if os.path.exists(SAVE_FILENAME):
        with open(SAVE_FILENAME, "r") as f:
            data = json.load(f)
            # convert saved dict back to ChatMessageHistory objects
            store_session_history = {}
            for session_id, messages_dict in data.items():
                history = ChatMessageHistory()  # type: ignore
                # messages_dict is list of message dicts (role, content)
                for msg, in messages_dict:
                    if msg["type"] == "human":
                        history.add_user_message(msg["data"]["content"])
                    elif msg["type"] == "ai":
                        history.add_ai_message(msg["data"]["content"])
                store_session_history[session_id] = history  # type: ignore
                return store_session_history    # type: ignore
    return {}   # type: ignore


def save_store():   # type: ignore
    data = {}
    for session_id, history in store_session_history.items():   # type: ignore
        # convert ChatMessageHistory to list of message dicts
        messages_dict = [message_to_dict(msg) for msg in history.messages]  # type: ignore
        data[session_id] = messages_dict
    with open(SAVE_FILENAME, "w") as f:
        json.dump(data, f, indent=2)

# load existing session history from file on startup
store_session_history = load_store()  # type: ignore

def get_session_history(session_id: str) -> Dict[str, str]:
    if session_id not in store_session_history:
        store_session_history[session_id] = ChatMessageHistory()
    return store_session_history[session_id]  # type: ignore


## 6. wrap the entire chain with message history

In [80]:
with_history_chain = RunnableWithMessageHistory(
    udf_chain,  # type: ignore
    get_session_history,  # type: ignore
    input_messages_key="input",
    history_messages_key="history"
)

7. execute in interactive loop

In [81]:
session_id = 'abc_898'
print(f"Bot ready with session_id: {session_id}. You can start chatting with the bot. Type 'exit', 'bye', or 'quit' to end the conversation.")
history = ""

while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "bye", "quit"]:
        break
    
    print(f"Processing user input: {user_input} with session_id: {session_id}")
    # history = get_session_history(session_id)
    bot_response = with_history_chain.invoke(  # type: ignore
        {"input": user_input}, 
        #  {"history": [history]},
         config={"configurable": {"session_id": session_id}}
        )
    
    print(f"\n\n\tBot Assistant: {bot_response.content}") # type: ignore
    

save_store()  # type: ignore
print("Session history saved. Goodbye!")
    # Update the session history
    # history.add_user_message(user_input)
    # history.add_ai_message(bot_response)


Bot ready with session_id: abc_898. You can start chatting with the bot. Type 'exit', 'bye', or 'quit' to end the conversation.
Processing user input: what is the radius of the earth with session_id: abc_898


	Bot Assistant: The radius of the Earth varies depending on where you measure it, as the Earth is not a perfect sphere but an **oblate spheroid** (slightly flattened at the poles and bulging at the equator). Here are the commonly used values:

1. **Equatorial radius**:  
   - Approximately **6,378.1 kilometers (3,963.2 miles)**.  
   - This is the distance from the Earth's center to the equator.

2. **Polar radius**:  
   - Approximately **6,356.8 kilometers (3,949.9 miles)**.  
   - This is the distance from the Earth's center to the poles.

3. **Mean radius** (average of the equatorial and polar radii):  
   - Approximately **6,371 kilometers (3,958.8 miles)**.  

The mean radius is often used in general calculations and discussions about the Earth's size.
Processing user input

## Key Takeaways from Project 20_1



    1. Memory requires explicit state management – LLMs are stateless. You must store and inject history manually. RunnableWithMessageHistory is the production pattern for this.

    2. Session identification is developer responsibility – Dev decide what a "session" is (user ID, conversation ID). The framework just stores/retrieves based on dev defined key.

    3. Prompt placeholder is mandatory – MessagesPlaceholder(variable_name="history") tells LangChain exactly where to inject the conversation history.

    4. Input dictionary shape matters – The chain expects {"input": str, "history": List[BaseMessage]}. RunnableWithMessageHistory handles the history key automatically.

    5. Simple LCEL (prompt | model) is enough for straightforward flows – No need for RunnableParallel, RunnableMap, or other advanced patterns until we're truly need parallel branches.

CLI is a valid test harness – We validated memory, session persistence, and basic interaction without any UI complexity.



Upgrating from 5 to 5A: will show case:

    I- Serialization – How to convert LangChain message objects to/from JSON.
    II- Persistence – Production apps need to store state outside memory.
    III- Multi-tenancy – Handling multiple users/sessions with a single bot.